In [6]:
  # ============================================================
# POST_TRAINING_TEST.ipynb
# Test the model using the testing apples
# ============================================================

import numpy as np
import pandas as pd
from scipy.io import loadmat
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import classification_report, confusion_matrix

print("======================================")
print("POST‑TRAINING TESTING NOTEBOOK")
print("======================================")

# ------------------------------------------------------------
# 1. LOAD FILES (Apple cubes + dry matter CSV)
# ------------------------------------------------------------
print("\n[1/4] Loading Apple.mat + CSV ...")

APPLE_MAT_PATH = "/content/drive/MyDrive/Colab Notebooks/dataset/Apple.mat"
CSV_PATH = "/content/drive/MyDrive/Colab Notebooks/dataset/SpectroFood_dataset.csv"

mat = loadmat(APPLE_MAT_PATH)

# Load apple cubes
NUM_APPLES = 240
NUM_BANDS = 141

apples = []
for i in range(1, NUM_APPLES + 1):
    name = f"A{i}"
    apples.append(mat[name].astype(np.float32))

csv = pd.read_csv(CSV_PATH)
dry = np.array([csv.loc[i, "Dry matter"] for i in range(NUM_APPLES)], dtype=np.float32)

print("✓ Loaded hypercubes + dry matter")


# ------------------------------------------------------------
# 2. REBUILD TEST FEATURES (same as training)
# ------------------------------------------------------------
print("\n[2/4] Building mean spectral vectors for each apple...")

X_all = []
for cube in apples:
    H, W, B = cube.shape
    pixels = cube.reshape(-1, B)
    X_all.append(pixels.mean(axis=0))

X_all = np.stack(X_all, axis=0)
print("✓ Feature matrix:", X_all.shape)


# ------------------------------------------------------------
# 3. LOAD TRAINING ARTIFACTS (SCALER + MODEL)
# ------------------------------------------------------------
print("\n[3/4] Loading trained model + scaler...")

# Load scaler parameters
scaler_mean = np.load("results_simple/scaler_mean.npy")
scaler_scale = np.load("results_simple/scaler_scale.npy")

X_scaled = (X_all - scaler_mean) / scaler_scale

# Load trained model
model = keras.models.load_model("/content/best_model.h5")

print("✓ Model loaded")
print("✓ Scaler loaded")


# ------------------------------------------------------------
# 4. RUN TESTING (predict labels for all apples)
# ------------------------------------------------------------
print("\n[4/4] Running predictions ...")

# Load maturity labels from training notebook (KMeans labels)
# You MUST load them from the CSV to ensure same mapping
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42)
cluster_ids = kmeans.fit_predict(dry.reshape(-1, 1))

# Sort so class 0 = least dry matter = unripe
cluster_means = [dry[cluster_ids == c].mean() for c in range(3)]
order = np.argsort(cluster_means)

true_labels = np.zeros_like(cluster_ids)
for new_label, old_cluster in enumerate(order):
    true_labels[cluster_ids == old_cluster] = new_label

# Predict
probs = model.predict(X_scaled)
pred_labels = np.argmax(probs, axis=1)

print("\nAccuracy on ALL apples:",
      np.mean(pred_labels == true_labels))

print("\nClassification Report:")
print(classification_report(true_labels, pred_labels,
                            target_names=["Unripe", "Medium", "Ripe"],
                            digits=4))

print("\nConfusion Matrix:")
print(confusion_matrix(true_labels, pred_labels))


POST‑TRAINING TESTING NOTEBOOK

[1/4] Loading Apple.mat + CSV ...
✓ Loaded hypercubes + dry matter

[2/4] Building mean spectral vectors for each apple...


✓ Feature matrix: (240, 141)

[3/4] Loading trained model + scaler...
✓ Model loaded
✓ Scaler loaded

[4/4] Running predictions ...
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 

Accuracy on ALL apples: 0.49166666666666664

Classification Report:
              precision    recall  f1-score   support

      Unripe     0.5000    0.2115    0.2973        52
      Medium     0.4859    0.6900    0.5702       100
        Ripe     0.5000    0.4318    0.4634        88

    accuracy                         0.4917       240
   macro avg     0.4953    0.4445    0.4437       240
weighted avg     0.4941    0.4917    0.4719       240


Confusion Matrix:
[[11 29 12]
 [ 5 69 26]
 [ 6 44 38]]
